# MLP Diagnosis – 430 Features
Predict if a patient is diseased (1) or healthy (0) from 430 MRI-derived numeric features.
Pipeline: load/augment data, feature engineering, optional Optuna search, final training,
evaluation, and artifact saving.


In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import optuna
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, roc_auc_score
from torch.utils.data import DataLoader, Dataset

from src.clinical_combat.robust.robust_MLP import (
    DEFAULT_MODEL_CONFIG,
    MODEL_DIR,
    PatientMLP,
    build_mlp_from_config,
    eval_epoch,
    fit
)
from src.clinical_combat.robust.robust_utils import remove_covariates_effects_metrics
from src.clinical_combat.robust.synthectic_sites_generations import (
    augment_df,
    generate_sites_no_file,
    split_train_test,
)

DATA_DIR = Path('DATA/processed/compilation/pairwise/mlp')
DATA_FILE = DATA_DIR / 'compilation.all_metrics.with_camcan.csv.gz'
SYNTHETIC_SITES_DIR = Path('DATA/processed/mlp_syn_sites')

DISEASE = 'ALL'
RUN_NAME = f'mlp_example'
SEED = 43
LOAD_DATA = False
DEVICE = 'cpu'

MODEL_DIR.mkdir(parents=True, exist_ok=True)


def set_seed(seed: int = 42):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)


## Usage & Configuration
- Adjust `MODEL_CFG` / `TRAIN_CFG` below for architecture and training settings.
- Toggle Optuna with `USE_OPTUNA`.
- Artifacts (weights, params, and optional generated datasets) are saved under
  `Pytorch_models/` and `DATA/processed/mlp_syn_sites/` using the `RUN_NAME` prefix.


In [ ]:
USE_OPTUNA = True

MODEL_CFG = DEFAULT_MODEL_CONFIG.copy()

TRAIN_CFG = {
    'batch_size': 64,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'epochs': 100,
    'patience': 10,
    'neg_weight': 10.0,          # class 0 weight in the loss
}

NEG_WEIGHT = float(TRAIN_CFG.get('neg_weight', 10.0))


In [ ]:
def show_class_balance(y):
    values, counts = np.unique(y, return_counts=True)
    for value, count in zip(values, counts):
        print(f'Class {int(value)}: {count}')


def plot_curves(train, val, ylabel='Loss'):
    plt.figure(figsize=(6, 4))
    epochs = range(1, len(train) + 1)
    plt.plot(epochs, train, label='train')
    plt.plot(epochs, val, label='val')
    plt.xlabel('Epoch')
    plt.ylabel(ylabel)
    plt.title(f'{ylabel} curve')
    plt.legend()
    plt.grid(True)
    plt.show()


def compute_zscore(df, value_col='mean_no_cov'):
    stats = (
        df.groupby('metric_bundle')[value_col]
        .agg(['mean', 'std'])
        .rename(columns={'mean': 'global_mean', 'std': 'global_std'})
    )
    stats['global_std'] = stats['global_std'].replace(0, 1e-6)
    df = df.merge(stats, on='metric_bundle', how='left')
    df['zscore'] = (df[value_col] - df['global_mean']) / df['global_std']
    return df.drop(columns=['global_mean', 'global_std'])


def gen_sites_for_mlp(
    df,
    sample_sizes=(100,),
    disease_ratios=(0.03, 0.1, 0.3, 0.5, 0.7, 0.8),
    num_tests_by_ratio=None,
    n_jobs=-1,
):
    tests_by_ratio = num_tests_by_ratio or {
        0.03: 200,
        0.10: 200,
        0.30: 67,
        0.50: 40,
        0.70: 29,
        0.80: 25,
    }
    frames = []
    for ratio in disease_ratios:
        num_tests = tests_by_ratio.get(ratio, 0)
        if num_tests == 0:
            continue
        sites = generate_sites_no_file(
            sample_sizes,
            [ratio],
            num_tests,
            df,
            disease=None,
            n_jobs=n_jobs,
        )
        for idx, site_df in enumerate(sites):
            site_df = site_df.copy()
            site_df['sid'] = site_df['sid'].astype(str) + f'_r{int(ratio * 100)}_t{idx}'
            adjusted = remove_covariates_effects_metrics(site_df)
            adjusted = compute_zscore(adjusted)
            frames.append(adjusted)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def build_feature_matrix(df, value_col='zscore', bundle_col='metric_bundle', healthy_tag='HC'):
    features = df.pivot(index='sid', columns=bundle_col, values=value_col)
    label = df.groupby('sid')['disease'].first().ne(healthy_tag).astype(int)
    matrix = features.assign(label=label).reset_index(drop=False)
    return matrix


def make_X_Y(df, value_col='zscore'):
    df = compute_zscore(df, value_col='mean_no_cov')
    matrix = build_feature_matrix(df, value_col=value_col)
    matrix = matrix.drop(columns=['sid'])
    X = matrix.drop(columns='label').values.astype(np.float32)
    y = matrix['label'].values.astype(np.float32)
    show_class_balance(y)
    return X, y


class PatientDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [ ]:
if LOAD_DATA:
    df_train = pd.read_csv(SYNTHETIC_SITES_DIR / f'{RUN_NAME}_train.csv.gz')
    df_val = pd.read_csv(SYNTHETIC_SITES_DIR / f'{RUN_NAME}_val.csv.gz')
    df_test = pd.read_csv(SYNTHETIC_SITES_DIR / f'{RUN_NAME}_test.csv.gz')
else:
    df_raw = pd.read_csv(DATA_FILE)
    if DISEASE != 'ALL':
        df_raw = df_raw[df_raw['disease'].isin([DISEASE, 'HC'])]
    df_raw = df_raw[
        ~((df_raw['disease'] == 'HC') & (df_raw['old_site'] != 'CamCAN'))
    ]
    df_raw = df_raw[
        ~df_raw['bundle'].isin(['left_ventricle', 'right_ventricle'])
    ].copy()
    print('Raw shape:', df_raw.shape)

    df_train, df_temp = split_train_test(df_raw, test_size=0.2, random_state=SEED)
    df_val, df_test = split_train_test(df_temp, test_size=0.5, random_state=SEED)

    df_train = augment_df(df_train, 5)
    df_train = gen_sites_for_mlp(df_train)
    print('Train after augmentation:', df_train.shape)

    df_val = augment_df(df_val, 8)
    df_val = gen_sites_for_mlp(df_val)
    print('Val after augmentation:', df_val.shape)

    df_test = augment_df(df_test, 8)
    df_test = gen_sites_for_mlp(df_test)
    print('Test after augmentation:', df_test.shape)


In [ ]:
if not LOAD_DATA:
    SYNTHETIC_SITES_DIR.mkdir(parents=True, exist_ok=True)
    for split_name, df_split in [('train', df_train), ('val', df_val), ('test', df_test)]:
        path = SYNTHETIC_SITES_DIR / f'{RUN_NAME}_{split_name}.csv.gz'
        df_split.to_csv(path, index=False, compression='gzip')
        print(f'Saved {split_name} -> {path} (shape={df_split.shape})')


In [ ]:
dupes = (
    df_train
    .groupby(['sid', 'metric_bundle'])
    .size()
    .loc[lambda s: s > 1]
)
print(f'Duplicate sid/metric_bundle pairs: {dupes.shape[0]}')

X_train, y_train = make_X_Y(df_train)
X_val, y_val = make_X_Y(df_val)
X_test, y_test = make_X_Y(df_test)
print('Shapes -> train:', X_train.shape, 'val:', X_val.shape, 'test:', X_test.shape)


In [ ]:
batch_size = int(TRAIN_CFG.get('batch_size', 64))
train_dl = DataLoader(PatientDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
val_dl = DataLoader(PatientDataset(X_val, y_val), batch_size=batch_size)
test_dl = DataLoader(PatientDataset(X_test, y_test), batch_size=batch_size)


In [ ]:
baseline = LogisticRegression(max_iter=1000, n_jobs=-1)
baseline.fit(X_train, y_train)
prob_val = baseline.predict_proba(X_val)[:, 1]
baseline_auc = roc_auc_score(y_val, prob_val)
print(f'Validation AUC (LogisticRegression): {baseline_auc:.3f}')


In [ ]:
study = None
if USE_OPTUNA:
    def objective(trial):
        hidden_dim1 = trial.suggest_int('h1', 128, 512, step=64)
        hidden_dim2 = trial.suggest_int('h2', 64, 256, step=32)
        hidden_dim3 = trial.suggest_int('h3', 32, 128, step=16)
        drop = trial.suggest_float('drop', 0.5, 0.7)
        lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
        wd = trial.suggest_float('wd', 1e-4, 1e-2, log=True)

        model = PatientMLP(hidden_dims=(hidden_dim1, hidden_dim2, hidden_dim3), drop=drop).to(DEVICE)
        _, _, _, best_auc = fit(
            model,
            train_dl,
            val_dl,
            epochs=30,
            lr=lr,
            wd=wd,
            patience=5,
            run_name='tune',
            device=DEVICE,
            neg_weight=NEG_WEIGHT,
        )
        return best_auc

    study = optuna.create_study(direction='maximize', pruner=optuna.pruners.MedianPruner())
    study.optimize(objective, n_trials=30, show_progress_bar=True)

    print('Best AUC:', study.best_value)
    print('Best params:', study.best_params)


In [ ]:
best_params = study.best_params if study is not None else None
hidden_dims = (
    [best_params['h1'], best_params['h2'], best_params['h3']]
    if best_params
    else MODEL_CFG['hidden_dims']
)

model_final = build_mlp_from_config({**MODEL_CFG, 'hidden_dims': hidden_dims}).to(DEVICE)
lr = float(best_params['lr']) if best_params else float(TRAIN_CFG['lr'])
wd = float(best_params.get('wd', TRAIN_CFG['weight_decay'])) if best_params else float(TRAIN_CFG['weight_decay'])

state, train_losses, val_losses, best_auc = fit(
    model_final,
    train_dl,
    val_dl,
    epochs=int(TRAIN_CFG['epochs']),
    lr=lr,
    wd=wd,
    patience=int(TRAIN_CFG['patience']),
    run_name=RUN_NAME,
    device=DEVICE,
    neg_weight=NEG_WEIGHT,
)


In [ ]:
plot_curves(train_losses, val_losses, ylabel='BCE Loss')


In [ ]:
test_loss, test_auc, test_f1, test_probs, test_labels = eval_epoch(
    model_final,
    test_dl,
    nn.BCEWithLogitsLoss(),
    device=DEVICE,
)
print(f'Test AUC: {test_auc:.3f} | Test F1: {test_f1:.3f}')

model_final.eval()
preds, labels = [], []
with torch.no_grad():
    for xb, yb in test_dl:
        preds.append(torch.sigmoid(model_final(xb.to(DEVICE))).cpu())
        labels.append(yb)

preds = torch.cat(preds).numpy()
labels = torch.cat(labels).numpy()
ConfusionMatrixDisplay.from_predictions(labels, preds > 0.5)
plt.show()


In [ ]:
torch.save(state, MODEL_DIR / f'{RUN_NAME}_weights.pt')

params_to_save = {**MODEL_CFG, 'hidden_dims': hidden_dims, 'lr': lr, 'weight_decay': wd}
with open(MODEL_DIR / f'{RUN_NAME}_params.json', 'w') as fp:
    json.dump(params_to_save, fp, indent=2)

print('Artifacts saved in', MODEL_DIR)


In [ ]:
sample = np.random.rand(430).reshape(1, -1)
with torch.no_grad():
    prob = torch.sigmoid(
        model_final(torch.tensor(sample, dtype=torch.float32).to(DEVICE))
    ).item()
print(f'Inference example – disease probability: {prob:.3f}')
